# 📊 Estadística descriptiva
**Modelos de Predicción Analítica · Diplomado en Machine Learning y Ciencia de Datos**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leslysandra/primeros-pasos-python/blob/main/notebooks/02_estadistica_descriptiva.ipynb)

**Autora:** Ing. Mgr. Lesly Zerna · **Versión:** 2.0 (Python)

> *"Antes de modelar, hay que entender."*

La estadística descriptiva **resume y describe las características principales de un conjunto de datos**. Es el primer paso **obligatorio** antes de cualquier modelo predictivo: revela problemas de calidad de datos, guía las decisiones de ingeniería de variables y fija expectativas realistas sobre el desempeño del modelo.

**Datasets:**
- `mtcars` — 32 autos con su consumo, peso y potencia (pequeño y fácil de revisar a ojo).
- `diamonds` — 53.940 diamantes con su precio y características (grande y más "real").

**Secciones**
1. Medidas de tendencia central
2. Medidas de dispersión
3. Forma: asimetría y curtosis
4. Tablas de frecuencia y proporciones
5. La familia `describe()`
6. Un perfil rápido de datos (el "`skim`" de Python)
7. Detectar valores faltantes
8. Detectar valores atípicos (*outliers*)
9. Análisis de correlación
10. Estadísticas descriptivas por grupo

> 💡 Las celdas **✏️ Tu turno** tienen espacios `____` para completar. Si aún no completas una, mostrará un error: es normal.

## Preparación: librerías y datos

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats                 # asimetría, curtosis, pruebas estadísticas
import seaborn as sns                       # gráficos estadísticos
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

# Función auxiliar para los ejercicios "✏️ Tu turno"
def revisar(condicion, ok="✅ ¡Correcto!", error="❌ Todavía no: revisa tu código e inténtalo otra vez."):
    """Imprime un mensaje según si tu respuesta es correcta."""
    print(ok if condicion else error)

In [ ]:
# mtcars: lo leemos desde un CSV; la columna "modelo" será el índice (los nombres de las filas)
URL_DATOS = "https://raw.githubusercontent.com/leslysandra/primeros-pasos-python/main/data/"
mtcars = pd.read_csv(URL_DATOS + "mtcars.csv", index_col="modelo")

# diamonds: viene incluido en la librería seaborn
diamonds = sns.load_dataset("diamonds")

mtcars.info()

In [ ]:
mtcars.head()

In [ ]:
diamonds.info()
diamonds.head()

## Sección 1 — Medidas de tendencia central

Describen el valor **"típico"** de una distribución. Usaremos `x`, el consumo en millas por galón (`mpg`) de `mtcars`.

In [ ]:
x = mtcars["mpg"]

# Media aritmética — sensible a valores atípicos
print("Media:   ", x.mean())

# Mediana — valor central, robusta a valores atípicos
print("Mediana: ", x.median())

In [ ]:
# Moda — el valor más frecuente. mode() puede devolver VARIOS valores si hay empate,
# por eso tomamos el primero con [0]
print("Moda de cilindros:", mtcars["cyl"].mode()[0])

In [ ]:
# Media recortada — descarta el 10 % más bajo y el 10 % más alto antes de promediar.
# Es más robusta que la media simple cuando hay asimetría o atípicos.
print("Media recortada:", stats.trim_mean(x, 0.10))

# Media ponderada — útil cuando las observaciones tienen distinta importancia
# (ejemplo: ponderar por el tamaño del motor, disp)
print("Media ponderada:", np.average(x, weights=mtcars["disp"]))

**¿Cuál usar?**

| Situación | Medida recomendada |
|---|---|
| Distribución simétrica | media ≈ mediana (cualquiera sirve) |
| Distribución asimétrica o con atípicos | **mediana** |
| Datos categóricos o discretos | **moda** |
| Encuestas con grupos de distinto peso | media ponderada |

### ✏️ Tu turno — Tendencia central

In [ ]:
# TODO: calcula la media y la mediana de `qsec` (tiempo en recorrer 1/4 de milla)
media_qsec = mtcars["____"].mean()
mediana_qsec = mtcars["qsec"].____()

print("Media:", round(media_qsec, 2), "| Mediana:", round(mediana_qsec, 2))
revisar(round(media_qsec, 2) == 17.85 and round(mediana_qsec, 2) == 17.71)

## Sección 2 — Medidas de dispersión

Indican **qué tan esparcidos** están los valores alrededor del centro. Un modelo entrenado con datos de poca variación aprende cosas muy distintas que uno entrenado con datos muy variables.

In [ ]:
print("Mínimo y máximo:", x.min(), x.max())
print("Rango (máx - mín):", np.ptp(x))   # ptp = "peak to peak"

In [ ]:
# Varianza — promedio de las desviaciones al cuadrado respecto de la media.
# pandas usa por defecto la fórmula MUESTRAL (divide entre n-1), igual que Excel y R.
print("Varianza:", x.var())

# Desviación estándar — en las mismas unidades que la variable
print("Desv. estándar:", x.std())

In [ ]:
# Coeficiente de variación (CV) — dispersión relativa, sin unidades
cv = x.std() / x.mean() * 100
print(f"CV de mpg: {cv:.1f} %")
# Regla práctica: CV < 15 % baja variabilidad; 15–30 % moderada; > 30 % alta

In [ ]:
# Rango intercuartílico (IQR) — rango del 50 % central, robusto a atípicos
print("IQR:", stats.iqr(x))

# Cuantiles
print(x.quantile([0, 0.25, 0.5, 0.75, 1]))   # mín, Q1, mediana, Q3, máx

In [ ]:
print(x.quantile([0.1, 0.9]))                      # percentiles 10 y 90
print(x.quantile(np.arange(0, 1.01, 0.2)))         # quintiles

In [ ]:
# Resumen de cinco números + media y más, en una sola línea
x.describe()

### ✏️ Tu turno — Dispersión

In [ ]:
# TODO: calcula el coeficiente de variación (%) de `hp` (caballos de fuerza)
# CV = desviación estándar / media * 100
cv_hp = mtcars["hp"].____() / mtcars["hp"].____() * 100

categoria = "baja" if cv_hp < 15 else ("moderada" if cv_hp <= 30 else "alta")
print(f"CV de hp: {cv_hp:.1f} % → variabilidad {categoria}")
revisar(round(cv_hp, 1) == 46.7 and categoria == "alta")

## Sección 3 — Forma: asimetría y curtosis

Muchos modelos (regresión lineal, Naive Bayes, PCA…) **suponen normalidad**. La asimetría y la curtosis miden qué tan lejos de la normal está una distribución.

### Asimetría (*skewness*)

| Valor | Interpretación |
|---|---|
| = 0 | simétrica (tipo normal) |
| > 0 | **sesgo a la derecha**: cola larga hacia valores altos; media > mediana |
| < 0 | **sesgo a la izquierda**: cola larga hacia valores bajos; media < mediana |
| \|valor\| > 1 | muy asimétrica: considera transformarla |

In [ ]:
print("Asimetría de mpg:", stats.skew(mtcars["mpg"]))   # sesgo leve a la derecha
print("Asimetría de hp: ", stats.skew(mtcars["hp"]))    # más asimétrica: autos de alta potencia alargan la cola

# Forma rápida de detectarlo: media - mediana (positivo = sesgo a la derecha)
print("Media - mediana de hp:", mtcars["hp"].mean() - mtcars["hp"].median())

> ℹ️ **Nota técnica:** `stats.skew()` usa la fórmula poblacional. `pandas` tiene `.skew()`, que aplica una corrección para muestras y da valores ligeramente distintos. Para interpretar (¿cerca de 0 o no?) cualquiera sirve; lo importante es **ser consistente**.

### Curtosis (*kurtosis*)

`stats.kurtosis()` devuelve la curtosis **en exceso** (ya le resta 3 a la de la normal):

| Valor | Interpretación |
|---|---|
| = 0 | como la normal |
| > 0 | **leptocúrtica**: colas pesadas y pico agudo |
| < 0 | **platicúrtica**: colas ligeras y pico plano |

In [ ]:
print("Curtosis (exceso) de mpg:", stats.kurtosis(mtcars["mpg"]))

### Transformaciones comunes para datos asimétricos

- Sesgo a la derecha (positivo): `log`, raíz cuadrada, `1/x`
- Sesgo a la izquierda (negativo): `x²`, `x³`

In [ ]:
hp_log = np.log(mtcars["hp"])     # transformación logarítmica
hp_sqrt = np.sqrt(mtcars["hp"])   # raíz cuadrada

print("Asimetría original:", round(stats.skew(mtcars["hp"]), 3))
print("Después del log:   ", round(stats.skew(hp_log), 3), "← mucho más cerca de 0")
print("Después de la raíz:", round(stats.skew(hp_sqrt), 3), "← mejora moderada")

In [ ]:
# Lo vemos en un gráfico: antes y después de la transformación
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.5))
sns.histplot(mtcars["hp"], bins=10, kde=True, ax=ejes[0]).set_title("hp original")
sns.histplot(hp_log, bins=10, kde=True, ax=ejes[1]).set_title("log(hp)")
plt.tight_layout()
plt.show()

### Prueba de normalidad de Shapiro-Wilk

Con **n < 5.000** observaciones. La hipótesis nula es que los datos vienen de una distribución normal:
- `p > 0,05` → no se puede rechazar la normalidad.
- `p < 0,05` → hay evidencia de que **no** es normal.

In [ ]:
print("mpg → p =", round(stats.shapiro(mtcars["mpg"]).pvalue, 4))   # > 0.05 → compatible con normal
print("hp  → p =", round(stats.shapiro(mtcars["hp"]).pvalue, 4))    # < 0.05 → evidencia de no normalidad

### ✏️ Tu turno — Forma

In [ ]:
# TODO: el precio (`price`) de los diamantes está muy sesgado a la derecha.
# 1) calcula su asimetría; 2) aplica logaritmo natural y vuelve a calcularla
asim_original = stats.skew(diamonds["price"])
asim_log = stats.skew(np.____(diamonds["____"]))

print("Antes:", round(asim_original, 2), "| Después del log:", round(asim_log, 2))
revisar(abs(asim_log) < 0.5 < asim_original)

## Sección 4 — Tablas de frecuencia y proporciones

Para variables **categóricas** no calculamos medias: **contamos frecuencias**.

In [ ]:
print(mtcars["cyl"].value_counts())                       # frecuencias absolutas
print(mtcars["cyl"].value_counts(normalize=True))         # proporciones
print((mtcars["cyl"].value_counts(normalize=True) * 100).round(1))   # porcentajes

**Tabla cruzada** (dos variables a la vez) con `pd.crosstab()`: cilindros × transmisión (`am`: 0 = automática, 1 = manual).

In [ ]:
print(pd.crosstab(mtcars["cyl"], mtcars["am"]))

# Con totales:
print(pd.crosstab(mtcars["cyl"], mtcars["am"], margins=True))

In [ ]:
# Porcentajes por FILA (cada fila suma 100 %): ¿qué % de los autos de 4 cilindros es manual?
(pd.crosstab(mtcars["cyl"], mtcars["am"], normalize="index") * 100).round(1)

In [ ]:
# Distribución de la calidad de corte (cut) de los diamantes, en porcentaje
(diamonds["cut"].value_counts(normalize=True) * 100).round(1)

In [ ]:
# Combinaciones corte × color, más frecuentes primero, con su porcentaje del total
frec = (
    diamonds
    .groupby(["cut", "color"], observed=True)
    .size()
    .reset_index(name="n")
    .assign(pct=lambda d: (d["n"] / d["n"].sum() * 100).round(2))
    .sort_values("n", ascending=False)
)
frec.head(10)

### ✏️ Tu turno — Frecuencias

In [ ]:
# TODO: crea una tabla cruzada de `gear` (número de marchas) por `am` (transmisión),
# con porcentajes por FILA
tabla = pd.crosstab(mtcars["____"], mtcars["____"], normalize="____")

print((tabla * 100).round(1))
revisar(tabla.shape == (3, 2) and abs(tabla.loc[4, 1] - 8 / 12) < 1e-9)

## Sección 5 — La familia `describe()`

In [ ]:
# El primer vistazo más rápido. .T transpone la tabla para que sea más fácil de leer
mtcars.describe().T

In [ ]:
# Para una sola columna
mtcars["hp"].describe()

In [ ]:
# Incluir también las columnas de texto/categóricas
diamonds.describe(include="all").T

Si necesitas **exactamente** ciertas estadísticas, usa `.agg()` con la lista de funciones que quieras:

In [ ]:
mtcars.agg(["mean", "std", "median", "min", "max"]).round(2).T

## Sección 6 — Un perfil rápido de datos: el "skim" de Python

En otros lenguajes existen funciones que entregan un **perfil completo** del dataset en una sola llamada (faltantes, tipos, media, desviación, mínimo, máximo…). En pandas podemos armarnos una propia. **No necesitas entender cómo está hecha por dentro:** ejecútala y reutilízala en tus proyectos.

> 💡 Existen también librerías externas (`skimpy`, `ydata-profiling`) que generan perfiles aún más completos si algún día los necesitas.

In [ ]:
def perfil(df):
    """Perfil rápido de un DataFrame: tipo, faltantes, valores únicos y estadísticas de las numéricas."""
    resumen = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "n_faltantes": df.isna().sum(),
        "pct_faltantes": (df.isna().mean() * 100).round(1),
        "n_unicos": df.nunique(),
    })
    numericas = df.select_dtypes("number").agg(["mean", "std", "min", "median", "max"]).T.round(2)
    return resumen.join(numericas)

perfil(mtcars)

In [ ]:
perfil(diamonds)

In [ ]:
# Estadísticas de varias columnas, agrupadas por una categoría
mtcars.groupby("cyl")[["mpg", "hp", "wt"]].describe().round(2)

## Sección 7 — Detectar valores faltantes

Los valores faltantes (`NaN`) rompen los modelos **en silencio**. Siempre audítalos primero.

Para practicar, **inyectamos faltantes** en una copia de `mtcars` (el original está completo):

In [ ]:
rng = np.random.default_rng(42)                 # generador aleatorio con semilla fija → reproducible

mtcars_na = mtcars.copy()
mtcars_na["hp"] = mtcars_na["hp"].astype(float)   # las columnas enteras no admiten NaN, las pasamos a decimal

filas_hp = rng.choice(len(mtcars_na), size=5, replace=False)
filas_mpg = rng.choice(len(mtcars_na), size=3, replace=False)

mtcars_na.iloc[filas_hp, mtcars_na.columns.get_loc("hp")] = np.nan
mtcars_na.iloc[filas_mpg, mtcars_na.columns.get_loc("mpg")] = np.nan

In [ ]:
# Total de faltantes en todo el dataset:
print("Total de NaN:", mtcars_na.isna().sum().sum())

# Faltantes por columna:
print(mtcars_na.isna().sum())

In [ ]:
# Porcentaje de faltantes por columna, en una tabla ordenada (solo las que tienen faltantes)
faltantes = pd.DataFrame({
    "n_faltantes": mtcars_na.isna().sum(),
    "pct_faltantes": (mtcars_na.isna().mean() * 100).round(1),
})
faltantes[faltantes["n_faltantes"] > 0].sort_values("n_faltantes", ascending=False)

In [ ]:
# ¿Qué FILAS tienen al menos un faltante?
mtcars_na[mtcars_na.isna().any(axis=1)]

### Patrones de datos faltantes

| Patrón | Significado | Gravedad |
|---|---|---|
| **MCAR** (*Missing Completely At Random*) | El faltante es puramente aleatorio | El más seguro: no introduce sesgo |
| **MAR** (*Missing At Random*) | El faltante se relaciona con **otras** variables observadas | Manejable |
| **MNAR** (*Missing Not At Random*) | El faltante depende del **propio valor** (ej.: quienes ganan más no declaran su ingreso) | El peor |

Veremos estrategias de imputación en `04_ingenieria_de_variables.ipynb`.

## Sección 8 — Detectar valores atípicos (*outliers*)

Los atípicos pueden distorsionar los coeficientes de una regresión, inflar el error del modelo y engañar los rankings de importancia de variables. **Identifícalos siempre antes de modelar y luego decide: eliminar, limitar (*cap*) o conservar.**

### Método 1: la regla del IQR (Tukey)

- Límite inferior = Q1 − 1,5 × IQR
- Límite superior = Q3 + 1,5 × IQR

Los valores fuera de esos límites se marcan como atípicos potenciales. Definimos una función para reutilizarla:

In [ ]:
def detectar_outliers_iqr(x, mult=1.5):
    """Devuelve una serie de True/False: True donde el valor es atípico según la regla del IQR."""
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    return (x < q1 - mult * iqr) | (x > q3 + mult * iqr)

marcas = detectar_outliers_iqr(mtcars["hp"])
mtcars.loc[marcas, ["hp", "cyl", "disp"]]   # las filas atípicas

In [ ]:
# Aplicada a TODAS las columnas numéricas: ¿cuántos atípicos hay en cada una?
mtcars.apply(detectar_outliers_iqr).sum().sort_values(ascending=False)

### Método 2: puntaje Z

Se marcan los valores a **más de 3 desviaciones estándar** de la media. Funciona mejor con distribuciones aproximadamente normales.

In [ ]:
z = stats.zscore(mtcars["hp"], ddof=1)   # estandariza: media 0, desviación 1
atipicos_z = np.abs(z) > 3

print("Atípicos por Z:", atipicos_z.sum())   # ninguno en este dataset pequeño
print("Z máximo:", round(np.abs(z).max(), 2))

### Método 3: visualización
En un diagrama de caja (*boxplot*), los atípicos aparecen como puntos individuales más allá de los bigotes.

In [ ]:
plt.figure(figsize=(4, 4))
sns.boxplot(y=mtcars["hp"], color="lightblue")
plt.title("Distribución de caballos de fuerza")
plt.ylabel("HP")
plt.show()

print("Valores atípicos:", mtcars.loc[marcas, "hp"].tolist())

### ✏️ Tu turno — Atípicos

In [ ]:
# TODO: usa la función detectar_outliers_iqr() con el precio (`price`) de los diamantes.
# ¿Cuántos diamantes tienen un precio atípico y qué porcentaje del total representan?
marcas_precio = detectar_outliers_iqr(diamonds["____"])

n_atipicos = marcas_precio.____()
pct_atipicos = marcas_precio.mean() * 100

print(f"{n_atipicos} diamantes atípicos ({pct_atipicos:.1f} %)")
revisar(n_atipicos == 3540)

## Sección 9 — Análisis de correlación

Antes de modelar, entiende cómo se relacionan las variables:
- **(a) con la variable objetivo** → ¿cuáles son más predictivas?
- **(b) entre sí** → la **multicolinealidad** puede volver inestable a un modelo.

In [ ]:
# Correlación de todas las variables con la variable objetivo (mpg), de mayor a menor
mtcars.corr()["mpg"].sort_values(ascending=False).round(2)

In [ ]:
# Matriz de correlación completa, visualizada como mapa de calor (solo el triángulo superior)
matriz = mtcars.corr()
mascara = np.tril(np.ones_like(matriz, dtype=bool), k=-1)   # oculta el triángulo inferior

plt.figure(figsize=(9, 7))
sns.heatmap(matriz, mask=mascara, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, annot_kws={"size": 8})
plt.title("Matriz de correlación de mtcars")
plt.show()

**Cómo interpretar la correlación de Pearson (r):**

| \|r\| | Interpretación |
|---|---|
| < 0,2 | despreciable |
| 0,2 – 0,4 | débil |
| 0,4 – 0,6 | moderada |
| 0,6 – 0,8 | fuerte |
| > 0,8 | muy fuerte → ¡posible multicolinealidad! |

**Advertencia de multicolinealidad:** `cyl`, `disp` y `wt` están correlacionadas a más de 0,8 entre sí. Incluir las tres en el mismo modelo puede causar inestabilidad.

In [ ]:
matriz.loc[["cyl", "disp", "wt"], ["cyl", "disp", "wt"]].round(2)

In [ ]:
# Prueba de significancia de la correlación: r, valor p e intervalo de confianza
res = stats.pearsonr(mtcars["mpg"], mtcars["wt"])

print("r =", round(res.statistic, 3))
print("p =", res.pvalue)
print("IC 95 %:", res.confidence_interval())

> ⚠️ **Correlación no implica causalidad.** Que el peso y el consumo estén fuertemente correlacionados no demuestra que el peso *cause* el consumo; solo que se mueven juntos.

### ✏️ Tu turno — Correlación

In [ ]:
# TODO: ¿qué variable tiene la correlación (en valor ABSOLUTO) más fuerte con `mpg`?
# Pasos: calcula las correlaciones con mpg, quita la propia mpg, toma el valor absoluto
# y pide el nombre de la variable con idxmax()
correlaciones = mtcars.corr()["mpg"].drop("mpg")

mas_fuerte = correlaciones.____().idxmax()
print("Variable más correlacionada con mpg:", mas_fuerte)
revisar(mas_fuerte == "wt")

## Sección 10 — Estadísticas descriptivas por grupo

Comparar distribuciones entre subgrupos suele ser la parte **más informativa** del análisis exploratorio, y revela patrones que ningún resumen global muestra.

In [ ]:
# Media y desviación de variables clave según el número de cilindros
(
    mtcars
    .groupby("cyl")
    .agg(
        n=("mpg", "size"),
        mpg_media=("mpg", "mean"),
        mpg_sd=("mpg", "std"),
        hp_media=("hp", "mean"),
        wt_media=("wt", "mean"),
    )
    .round(2)
)

In [ ]:
# Precio de los diamantes según la calidad del corte
(
    diamonds
    .groupby("cut", observed=True)
    .agg(
        n=("price", "size"),
        precio_medio=("price", "mean"),
        precio_mediana=("price", "median"),
        precio_sd=("price", "std"),
        pct_sobre_5k=("price", lambda s: (s > 5000).mean() * 100),
    )
    .round(1)
    .sort_values("precio_medio", ascending=False)
)

> 🤔 **Observa algo curioso:** los diamantes de corte "Ideal" (el mejor) tienen el precio medio **más bajo**. ¿Por qué? Porque el corte no es la única variable que influye: los diamantes "Ideal" suelen ser más **pequeños** (menos quilates). Es un buen ejemplo de por qué una sola variable no cuenta toda la historia, y de por qué necesitaremos modelos con **varias variables a la vez**.

### ANOVA: ¿las medias de los grupos son significativamente distintas?

(Lo veremos en profundidad en el módulo de modelado; aquí sirve de contexto.)

In [ ]:
modelo_anova = smf.ols("mpg ~ C(cyl)", data=mtcars).fit()   # C(...) indica que cyl es una categoría
sm.stats.anova_lm(modelo_anova)
# Valor p (PR(>F)) muy pequeño → los grupos de cilindros tienen consumos medios distintos → cyl sirve como predictora

### ✏️ Tu turno — Estadísticas por grupo

In [ ]:
# TODO: calcula el consumo medio (mpg) de los autos según su transmisión (am: 0 = automática, 1 = manual)
mpg_por_transmision = mtcars.groupby("____")["mpg"].____()

print(mpg_por_transmision.round(2))
revisar(round(mpg_por_transmision[1], 2) == 24.39 and round(mpg_por_transmision[0], 2) == 17.15)

## ✅ Lista de verificación de estadística descriptiva
*(úsala al inicio de cada proyecto)*

- [ ] `df.info()`, `df.describe()` o `perfil(df)` → perfil general de los datos
- [ ] Dimensiones (`shape`) y tipos de variables
- [ ] **Valores faltantes:** ¿qué columnas?, ¿qué porcentaje?
- [ ] **Variables numéricas:** media, mediana, desviación, asimetría, atípicos
- [ ] **Variables categóricas:** tablas de frecuencia, categorías raras
- [ ] **Variable objetivo:** su distribución y, si es clasificación, su balance entre clases
- [ ] **Correlaciones:** con la variable objetivo y entre predictoras
- [ ] **Comparaciones por grupo:** medias según categoría
- [ ] **Patrones en el tiempo:** si hay una columna de fecha, grafícala

---

### 🎉 Fin del notebook de estadística descriptiva
**Siguiente:** `03_visualizacion.ipynb`